In [1]:
import warnings

In [2]:
!pip install duckdb pandas


In [3]:
import os
import duckdb

try:
    # 1. Attempt to import the Google Colab specific library
    from google.colab import drive
    print("☁ Environment Detected: Google Colab (Cloud Sandbox)")
    
    # 2. Define cloud-based directory locations
    DATA_DIR = "./data"
    DB_PATH = "./credit_risk_warehouse.db"
    
except ImportError:
    # 3. Fallback execution if running on your local machine
    print("💻 Environment Detected: VS Code (Personal Hard Drive)")
    
    # 4. Define local relative path routing
    DATA_DIR = "../data"
    DB_PATH = "../credit_risk_warehouse.db"

# 5. Establish a live, persistent link to our database file
conn = duckdb.connect(DB_PATH)

# 6. Print physical absolute validation paths to the screen
print(f"✔ Active Warehouse File: {os.path.abspath(DB_PATH)}")
print(f"✔ Targeted Data Folder: {os.path.abspath(DATA_DIR)}")

💻 Environment Detected: VS Code (Personal Hard Drive)
✔ Active Warehouse File: c:\Users\HP\OneDrive\Desktop\Thrive Africa\Advanced Data Science and Analytics\credit-risk-pipeline\credit_risk_warehouse.db
✔ Targeted Data Folder: c:\Users\HP\OneDrive\Desktop\Thrive Africa\Advanced Data Science and Analytics\credit-risk-pipeline\data


In [4]:
import os
import glob 

# Looks for all CSV files in the specified data directory and prints their paths
search_pattern = os.path.join(DATA_DIR, "*.csv")
csv_files = glob.glob(search_pattern)

print(f"Searching for new data files in: {os.path.abspath(DATA_DIR)}")
print(f"Found {len(csv_files)} CSV files:")

#Loops through every single file found and registers it into DuckDB warehouse
for file_path in csv_files:
    #Extracts the clean file name without extension to use as a database table name
    base_name = os.path.basename(file_path)
    table_name = os.path.splitext(base_name)[0].lower()  # Converts to lowercase for consistency

    if "description" in table_name:
        warnings.warn(f"⚠ Skipping documentation file: '{base_name}' (Not needed in database as it contains 'description' in its name).")
        continue                        # this skips the current iteration and moves to the next file in the loop
    print(f"Loading {base_name} into table '{table_name}'...")

    # SQL command to create or replace a table in DuckDB and load data from the CSV file
    conn.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_csv_auto('{file_path}');
    """)

    # Let's verify row ingestion by counting the number of rows in the newly created table
    row_count = conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f" Suceessfully loaded '{table_name}' with {row_count:,} rows.")


print("\n ✔ All relational tables have been successfully loaded into the DuckDB analytical warehouse.")


Searching for new data files in: c:\Users\HP\OneDrive\Desktop\Thrive Africa\Advanced Data Science and Analytics\credit-risk-pipeline\data
Found 10 CSV files:
Loading application_test.csv into table 'application_test'...
 Suceessfully loaded 'application_test' with 48,744 rows.
Loading application_train.csv into table 'application_train'...
 Suceessfully loaded 'application_train' with 307,511 rows.
Loading bureau.csv into table 'bureau'...
 Suceessfully loaded 'bureau' with 1,716,428 rows.
Loading bureau_balance.csv into table 'bureau_balance'...
 Suceessfully loaded 'bureau_balance' with 27,299,925 rows.
Loading credit_card_balance.csv into table 'credit_card_balance'...
 Suceessfully loaded 'credit_card_balance' with 3,840,312 rows.
Loading installments_payments.csv into table 'installments_payments'...


C:\Users\HP\AppData\Local\Temp\ipykernel_26020\3700743817.py:18: UserWarning: ⚠ Skipping documentation file: 'HomeCredit_columns_description.csv' (Not needed in database as it contains 'description' in its name).
  warnings.warn(f"⚠ Skipping documentation file: '{base_name}' (Not needed in database as it contains 'description' in its name).")


 Suceessfully loaded 'installments_payments' with 13,605,401 rows.
Loading POS_CASH_balance.csv into table 'pos_cash_balance'...
 Suceessfully loaded 'pos_cash_balance' with 10,001,358 rows.
Loading previous_application.csv into table 'previous_application'...
 Suceessfully loaded 'previous_application' with 1,670,214 rows.
Loading sample_submission.csv into table 'sample_submission'...
 Suceessfully loaded 'sample_submission' with 48,744 rows.

 ✔ All relational tables have been successfully loaded into the DuckDB analytical warehouse.


In [5]:
print("== MANDATORY DATA QUALITY AUDIT ==")

# Total Row Count Check
# -------------------------------------------------------------
print("\n CHECK 1: Verifying total rows in parent table...")
row_count_query = conn.execute("SELECT COUNT(*) FROM application_train;").fetchone()
total_rows = row_count_query[0]

print(f" Total Rows Found in 'application_train': {total_rows:,}")


# 2. Null Value Audit on Primary Key
# -------------------------------------------------------------
print("\n CHECK 2: Auditing for missing Primary Keys...")
null_query = conn.execute("""
    SELECT COUNT(*) 
    FROM application_train 
    WHERE SK_ID_CURR IS NULL;
""").fetchone()
null_count = null_query[0]

print(f" Missing (Null) Primary Keys: {null_count}")
if null_count == 0:
    print(" PASS: No missing primary keys detected.")
else:
    print(" FAIL: Missing IDs found. Pipeline needs investigation.")


# 3. Duplicate Record Audit
# -------------------------------------------------------------
print("\n CHECK 3: Asserting Primary Key uniqueness...")
duplicate_query = conn.execute("""
    SELECT 
        COUNT(*) as total_records, 
        COUNT(DISTINCT SK_ID_CURR) as unique_applicants 
    FROM application_train;
""").fetchone()

total_records, unique_applicants = duplicate_query

print(f" Total Rows: {total_records:,} | Unique Applicant IDs: {unique_applicants:,}")
if total_records == unique_applicants:
    print(" PASS: 'SK_ID_CURR' is perfectly unique. The table grain is correct.")
else:
    print(" FAIL: Duplicate applicant IDs found! Structural integrity compromised.")

SyntaxError: invalid syntax (1382555588.py, line 7)

In [ ]:
print("=== 🚀 EXECUTING STRATEGIC SQL PIPELINE ===")

# Read the standalone SQL file contents directly into Python
with open("../sql/01_staging.sql", "r") as f:
    staging_sql_query = f.read()

# Execute the query script file through our DuckDB connection pipe
conn.execute(staging_sql_query)

print("✅ SUCCESS: '01_staging.sql' executed. 'mart_applicant_features' created!")